# Hexapod: training the wheel-legged locomotion policy (MuJoCo MJX + Brax PPO)

Trains the policy from scratch: 300,000,000 environment steps.

- When it finishes, download `policy.npz` and `hexapod.mp4` from the **Output** tab.
   Copy `policy.npz` to `models/policy.npz` in the repository and run `python play.py`.

In [ ]:
# Kaggle's preinstalled JAX does not work with Brax 0.14: install the tested versions.
!pip uninstall -q -y jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt jax-cuda13-plugin jax-cuda13-pjrt
!rm -rf /usr/local/lib/python3*/dist-packages/jax_plugins/xla_cuda13 /usr/local/lib/python3*/dist-packages/jax_cuda13*
!pip install -q "jax[cuda12]==0.6.2" "brax==0.14.1" "mujoco==3.17.0" "mujoco-mjx==3.17.0" "flax==0.11.2" mediapy

In [ ]:
!mkdir -p hexapod assets

In [ ]:
%%writefile hexapod/__init__.py
"""Hexapod robot: model, environments, policy and on-screen overlay."""

In [ ]:
%%writefile hexapod/core.py
"""Shared robot, terrain, observation and reward definitions (NumPy and JAX)."""
from pathlib import Path
from typing import NamedTuple

import numpy as np

MODEL_PATH = Path(__file__).resolve().parent.parent / "assets" / "hexapod.xml"

LEGS = ("LF", "LM", "LR", "RF", "RM", "RR")
WHEEL_LEGS = ("LF", "LR", "RF", "RR")
JOINTS = tuple(f"{leg}_{seg}" for leg in LEGS for seg in ("coxa", "femur", "tibia"))
WHEELS = tuple(f"{leg}_wheel" for leg in WHEEL_LEGS)
ACTUATORS = JOINTS + WHEELS
N_JOINTS, N_WHEELS, N_ACT, N_LEGS = len(JOINTS), len(WHEELS), len(ACTUATORS), len(LEGS)
ACT_LEG = np.array([i // 3 for i in range(N_JOINTS)] + [LEGS.index(leg) for leg in WHEEL_LEGS])
CONTACT_GEOMS = tuple(f"{leg}_tire" if leg in WHEEL_LEGS else f"{leg}_foot" for leg in LEGS)
WHEEL_LEG_IDX = np.array([LEGS.index(leg) for leg in WHEEL_LEGS])
HAS_WHEEL = np.array([leg in WHEEL_LEGS for leg in LEGS])
LEG_WHEEL = np.array([WHEEL_LEGS.index(leg) if leg in WHEEL_LEGS else 0 for leg in LEGS])
IS_WHEEL = np.arange(N_ACT) >= N_JOINTS

CTRL_DT = 0.02
N_SUBSTEPS = 5
KP, KD = 10.0, 0.1
WHEEL_KV = 0.05
WHEEL_SPEED_MAX = 35.0
TORQUE_LIMITS = np.array([2.0] * N_JOINTS + [0.6] * N_WHEELS)
NO_LOAD_SPEED = np.array([8.0] * N_JOINTS + [45.0] * N_WHEELS)
ACTION_SCALE = 0.6
SPAWN_HEIGHT = 0.145
NOMINAL_HEIGHT = 0.13
EPISODE_STEPS = 1000

TILE = 2.0
LEVELS = 8
TERRAIN_TYPES = ("gravel", "ramp", "bumps", "rubble", "blocks", "stairs", "trenches", "mixed")
N_TYPES = len(TERRAIN_TYPES)
START_X = 0.5
BORDER = 1.0
INIT_LEVEL_MAX = 2
SPAWN_NOISE = 0.15
P_EDGE_SPAWN = 0.5
EDGE_OFFSET = 0.9
PROMOTE_DIST = 1.0
DEMOTE_RATIO = 0.25

CMD_LOW = np.array([-0.5, -0.2, -1.2])
CMD_HIGH = np.array([1.3, 0.2, 1.2])
CMD_SCALE = 1.0 / np.maximum(-CMD_LOW, CMD_HIGH)
SPEED_DROP = 0.08
CMD_HOLD = (200, 300)
CMD_SEGMENTS = EPISODE_STEPS // CMD_HOLD[0] + 1
P_STAND = 0.1

HEALTHY, LIMP, LOCKED, WEAK = 0, 1, 2, 3
FAIL_P = np.array([0.25, 0.15, 0.20, 0.10, 0.25, 0.05])
FAIL_CUM = np.cumsum(FAIL_P)
NONE, ONE_MOTOR, ONE_LEG, TWO_LEGS, ALL_WHEELS, WHEELS_AND_LEG = range(6)
P_LIMP, P_LOCKED = 0.5, 0.25
P_WHEELS_LIMP = 0.5
WEAK_RANGE = (0.3, 0.6)
FAIL_EARLIEST, FAIL_LATEST = 50, 750
NEVER = 10**6
GEAR_FRICTION = np.array([0.08] * N_JOINTS + [0.005] * N_WHEELS)
GEAR_DAMPING = np.array([0.02] * N_JOINTS + [0.002] * N_WHEELS)

FRICTION_RANGE = (0.6, 1.2)
SOFTNESS_RANGE = (0.01, 0.03)

SCAN_X = np.round(np.arange(-0.2, 0.401, 0.05), 3)
SCAN_Y = np.round(np.arange(-0.2, 0.201, 0.1), 3)
SCAN_POINTS = np.array([(x, y) for x in SCAN_X for y in SCAN_Y])
N_SCAN = len(SCAN_POINTS)
FOOTPRINT = np.flatnonzero((np.abs(SCAN_POINTS[:, 0]) < 0.151) & (np.abs(SCAN_POINTS[:, 1]) < 0.101))
SPAWN_GRID = np.array([(x, y) for x in np.linspace(-0.2, 0.2, 5) for y in np.linspace(-0.2, 0.2, 5)])
NORMAL_SPAN = 0.12
CONTACT_TOL = 0.006

GAIT_FREQ = (1.5, 2.0)
GAIT_FAST = 0.45
WALK_MAX = 0.5
GAIT_OFFSET = np.array([0.0 if leg in ("LF", "RM", "LR") else 0.5 for leg in LEGS])
GAIT_MARGIN = 0.05
SWING_HEIGHT = 0.05
HOP_MIN = 0.2

VEL_NOISE = 0.05
HISTORY = 10
ACTOR_OBS = 3 + 3 + 3 + 2 * N_JOINTS + N_WHEELS + N_ACT + 3 + 2
HIST_SIZE = ACTOR_OBS * HISTORY
OBS_SIZE = HIST_SIZE + N_SCAN
CRITIC_OBS = ACTOR_OBS + N_SCAN + 3 + 1 + 3 + 3 * N_ACT + 2 * N_LEGS
REWARD_KEYS = ("track_lin", "track_yaw", "yaw_drift", "upright", "bounce", "smooth", "effort", "alive", "wheels_down",
               "gait", "clearance", "wheel_hop", "posture", "height", "stand_still", "stand_pose", "stand_feet")


class Terrain(NamedTuple):
    heights: object
    x0: float
    y0: float
    cell: float


def addresses(model):
    for i, name in enumerate(ACTUATORS):
        assert model.actuator(i).trnid[0] == model.joint(name).id, f"actuator {i} does not drive joint {name}"
    qadr = np.array([model.joint(name).qposadr[0] for name in JOINTS])
    vadr = np.array([model.joint(name).dofadr[0] for name in JOINTS])
    wadr = np.array([model.joint(name).dofadr[0] for name in WHEELS])
    contact = np.array([model.geom(name).id for name in CONTACT_GEOMS])
    return qadr, vadr, wadr, contact, model.geom_size[contact, 0].copy()


def load_terrain(model):
    gid = model.geom("terrain").id
    hid = model.geom_dataid[gid]
    nrow, ncol = int(model.hfield_nrow[hid]), int(model.hfield_ncol[hid])
    size, pos, adr = model.hfield_size[hid], model.geom_pos[gid], model.hfield_adr[hid]
    heights = model.hfield_data[adr:adr + nrow * ncol].reshape(nrow, ncol) * size[2] + pos[2]
    cell = 2.0 * size[0] / (ncol - 1)
    assert abs(2.0 * size[1] / (nrow - 1) - cell) < 1e-9, "terrain cells must be square"
    assert abs(pos[0] - size[0] - (START_X - BORDER)) < 1e-6 and abs(2 * size[0] - LEVELS * TILE - 2 * BORDER) < 1e-6 \
        and abs(2 * size[1] - N_TYPES * TILE - 2 * BORDER) < 1e-6, "hexapod.xml terrain does not match the layout in hexapod/core.py"
    return Terrain(np.asarray(heights, np.float64), float(pos[0] - size[0]), float(pos[1] - size[1]), float(cell))


def tile_center(level, column):
    return START_X + (level + 0.5) * TILE, (column + 0.5 - N_TYPES / 2) * TILE


def ground_height(xp, terrain, x, y):
    h = terrain.heights
    nrow, ncol = h.shape
    fx = xp.clip((x - terrain.x0) / terrain.cell, 0.0, ncol - 1.001)
    fy = xp.clip((y - terrain.y0) / terrain.cell, 0.0, nrow - 1.001)
    ix, iy = xp.floor(fx).astype(xp.int32), xp.floor(fy).astype(xp.int32)
    tx, ty = fx - ix, fy - iy
    return ((1 - ty) * ((1 - tx) * h[iy, ix] + tx * h[iy, ix + 1])
            + ty * ((1 - tx) * h[iy + 1, ix] + tx * h[iy + 1, ix + 1]))


def heading(xp, q):
    w, x, y, z = q[0], q[1], q[2], q[3]
    return xp.arctan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))


def scan_heights(xp, terrain, qpos):
    yaw = heading(xp, qpos[3:7])
    c, s = xp.cos(yaw), xp.sin(yaw)
    px = qpos[0] + c * SCAN_POINTS[:, 0] - s * SCAN_POINTS[:, 1]
    py = qpos[1] + s * SCAN_POINTS[:, 0] + c * SCAN_POINTS[:, 1]
    return ground_height(xp, terrain, px, py)


def ground_level(xp, scan):
    return xp.mean(scan[FOOTPRINT])


def ground_normal(xp, terrain, x, y):
    d = NORMAL_SPAN
    h = ground_height(xp, terrain, x + np.array([d, -d, 0.0, 0.0]), y + np.array([0.0, 0.0, d, -d]))
    gx, gy = (h[0] - h[1]) / (2 * d), (h[2] - h[3]) / (2 * d)
    inv = 1.0 / xp.sqrt(1.0 + gx * gx + gy * gy)
    return xp.stack([-gx * inv, -gy * inv, inv])


def spawn_height(xp, terrain, x, y):
    return xp.max(ground_height(xp, terrain, x + SPAWN_GRID[:, 0], y + SPAWN_GRID[:, 1])) + SPAWN_HEIGHT


def foot_gaps(xp, terrain, pos, radius):
    return pos[:, 2] - radius - ground_height(xp, terrain, pos[:, 0], pos[:, 1])


def quat_to_mat(xp, q):
    w, x, y, z = q[0], q[1], q[2], q[3]
    return xp.stack([
        xp.stack([1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)]),
        xp.stack([2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)]),
        xp.stack([2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)]),
    ])


def command_at(xp, cmd_seq, cmd_t, t):
    return cmd_seq[xp.sum(cmd_t <= t) - 1]


def active_modes(xp, mode, fail_t, t):
    return xp.where(t >= fail_t, mode, HEALTHY)


def leg_roles(xp, mode):
    working = (mode == HEALTHY) | (mode == WEAK)
    joints_ok = xp.all(working[:N_JOINTS].reshape(N_LEGS, 3), axis=1)
    wheel_ok = working[N_JOINTS:][LEG_WHEEL] & HAS_WHEEL
    return joints_ok & wheel_ok, joints_ok & ~wheel_ok


def actuator_torque(xp, action, q, qd, w, mode, strength, q_lock):
    target = xp.where(mode[:N_JOINTS] == LOCKED, q_lock, action[:N_JOINTS] * ACTION_SCALE)
    wheel_target = xp.where(mode[N_JOINTS:] == LOCKED, 0.0, action[N_JOINTS:] * WHEEL_SPEED_MAX)
    tau = xp.concatenate([KP * (target - q) - KD * qd, WHEEL_KV * (wheel_target - w)])
    speed = xp.concatenate([qd, w])
    limit = TORQUE_LIMITS * strength
    tau = xp.clip(tau, -limit * xp.clip(1.0 + speed / NO_LOAD_SPEED, 0.0, 1.0),
                  limit * xp.clip(1.0 - speed / NO_LOAD_SPEED, 0.0, 1.0))
    unpowered = -GEAR_FRICTION * xp.tanh(speed / 0.05) - GEAR_DAMPING * speed
    return xp.where(mode == LIMP, unpowered, tau)


def park_brake(xp, action, cmd):
    standing = xp.sum(xp.abs(cmd)) < 0.05
    return xp.where(standing & IS_WHEEL, 0.0, action)


def gait_step(xp, phase, cmd):
    speed = xp.sqrt(xp.square(cmd[0]) + xp.square(cmd[1]))
    freq = GAIT_FREQ[0] + (GAIT_FREQ[1] - GAIT_FREQ[0]) * xp.clip(speed / GAIT_FAST, 0.0, 1.0)
    moving = xp.sum(xp.abs(cmd)) >= 0.05
    return xp.where(moving, (phase + freq * CTRL_DT) % 1.0, phase)


def air_time_step(xp, air_time, last_contact, contact):
    contact_filt = contact | last_contact
    first = (air_time > 0.0) & contact_filt
    air_time = air_time + CTRL_DT
    landed = xp.where(first, air_time, 0.0)
    return xp.where(contact_filt, 0.0, air_time), landed, first


def actor_obs(xp, qpos, qvel, qadr, vadr, wadr, last_action, cmd, phase, vel_noise=0.0):
    rot = quat_to_mat(xp, qpos[3:7])
    angle = 2.0 * np.pi * phase
    return xp.concatenate([
        2.0 * (rot.T @ qvel[0:3] + vel_noise),
        0.25 * qvel[3:6],
        -rot[2],
        qpos[qadr],
        0.05 * qvel[vadr],
        0.02 * qvel[wadr],
        last_action,
        cmd * CMD_SCALE,
        xp.stack([xp.sin(angle), xp.cos(angle)]),
    ])


def scan_obs(xp, qpos, scan):
    return 5.0 * xp.clip(qpos[2] - scan - NOMINAL_HEIGHT, -0.3, 0.3)


def critic_obs(xp, actor_now, scan_o, qpos, qvel, ground, normal, mode, strength, contact, air_time):
    rot = quat_to_mat(xp, qpos[3:7])
    return xp.concatenate([
        actor_now,
        scan_o,
        2.0 * (rot.T @ qvel[0:3]),
        5.0 * (qpos[2:3] - ground),
        rot.T @ normal,
        xp.where(mode == LIMP, 1.0, 0.0),
        xp.where(mode == LOCKED, 1.0, 0.0),
        strength,
        xp.where(contact, 1.0, 0.0),
        5.0 * air_time,
    ])


def body_velocity(xp, qpos, qvel):
    v = quat_to_mat(xp, qpos[3:7]).T @ qvel[0:3]
    return xp.concatenate([v[:2], qvel[5:6]])


def reward_terms(xp, qpos, qvel, qadr, vadr, wadr, cmd, action, last_action, tau, mode,
                 ground, normal, contact, landed, first, phase, gaps):
    rot = quat_to_mat(xp, qpos[3:7])
    v = rot.T @ qvel[0:3]
    w = qvel[3:6]
    lin_tol = 0.02 + 0.05 * (xp.square(cmd[0]) + xp.square(cmd[1]))
    yaw_tol = 0.05 + 0.1 * xp.square(cmd[2])
    yaw_err = xp.abs(w[2] - cmd[2])
    rolling, walking = leg_roles(xp, mode)
    usable = rolling | walking
    working = (mode == HEALTHY) | (mode == WEAK)
    moving = xp.sum(xp.abs(cmd)) >= 0.05
    fidget = xp.sum(xp.square(qvel[vadr])) + 0.5 * xp.sum(xp.square(qvel[wadr]))
    off_pose = xp.sum(xp.where(working[:N_JOINTS], xp.abs(qpos[qadr]), 0.0))
    lifted = xp.sum(xp.where(usable & ~contact, 1.0, 0.0))
    speed_cmd = xp.sqrt(xp.square(cmd[0]) + xp.square(cmd[1]))
    no_rolling = xp.sum(xp.where(rolling, 1.0, 0.0)) < 0.5
    rhythmic = walking & moving & (HAS_WHEEL | (speed_cmd <= WALK_MAX) | no_rolling)
    leg_phase = (phase + GAIT_OFFSET) % 1.0
    swing = leg_phase < 0.5
    certain = xp.abs(leg_phase % 0.5 - 0.25) < 0.25 - GAIT_MARGIN
    in_rhythm = xp.where(swing, ~contact, contact)
    gait = xp.sum(xp.where(rhythmic & certain, xp.where(in_rhythm, 1.0, -1.0), 0.0))
    clearance = xp.sum(xp.where(rhythmic & certain & swing, xp.clip(gaps, 0.0, SWING_HEIGHT) / SWING_HEIGHT, 0.0))
    hops = xp.where(rolling & first, xp.clip(HOP_MIN - landed, 0.0, HOP_MIN), 0.0)
    return {
        "track_lin": 2.0 * xp.exp(-(xp.square(v[0] - cmd[0]) + xp.square(v[1] - cmd[1])) / lin_tol),
        "track_yaw": 1.5 * xp.exp(-xp.square(yaw_err) / yaw_tol),
        "yaw_drift": -0.5 * yaw_err,
        "upright": -2.0 * (1.0 - rot[:, 2] @ normal),
        "bounce": -1.0 * xp.square(v[2]) - 0.05 * (xp.square(w[0]) + xp.square(w[1])),
        "smooth": -0.02 * xp.sum(xp.square(action - last_action)),
        "effort": -2e-3 * xp.sum(xp.square(tau)),
        "alive": 0.25 + 0.0 * rot[2, 2],
        "wheels_down": 0.1 * xp.sum(xp.where(rolling & contact, 1.0, 0.0)),
        "gait": 0.3 * gait,
        "clearance": 0.1 * clearance,
        "wheel_hop": -1.0 * xp.where(moving, xp.sum(hops), 0.0),
        "posture": -0.1 * xp.sum(xp.where(working[:N_JOINTS], xp.square(qpos[qadr]), 0.0)),
        "height": -20.0 * xp.square(qpos[2] - ground - NOMINAL_HEIGHT),
        "stand_still": -0.02 * xp.where(moving, 0.0, fidget),
        "stand_pose": -1.0 * xp.where(moving, 0.0, off_pose),
        "stand_feet": -0.5 * xp.where(moving, 0.0, lifted),
    }


def fallen(xp, qpos, ground):
    up = 1.0 - 2.0 * (xp.square(qpos[4]) + xp.square(qpos[5]))
    return (qpos[2] - ground < 0.05) | (up < 0.5)

In [ ]:
%%writefile hexapod/model.py
"""Generates hexapod.xml and the terrain (terrain.bin, terrain_lite.bin, terrain_tex.png).

    python -m hexapod.model
"""
import hashlib
import math
import struct
import zlib
from pathlib import Path

import numpy as np

from .core import BORDER, LEVELS, START_X, TERRAIN_TYPES, TILE

ASSETS = Path(__file__).resolve().parent.parent / "assets"
OUT = ASSETS / "hexapod.xml"
TERRAIN_FILE = ASSETS / "terrain.bin"
TERRAIN_LITE_FILE = ASSETS / "terrain_lite.bin"
TEXTURE_FILE = ASSETS / "terrain_tex.png"
RES = TILE / 66
MARGIN = 0.15
WHEEL_RADIUS = 0.03
WHEEL_DROP = 0.102
WHEEL_OFFSET = 0.022

LEGS = (
    ("LF", (0.12, 0.08), 0.7854, True),
    ("LM", (0.0, 0.09), 1.5708, False),
    ("LR", (-0.12, 0.08), 2.3562, True),
    ("RF", (0.12, -0.08), -0.7854, True),
    ("RM", (0.0, -0.09), -1.5708, False),
    ("RR", (-0.12, -0.08), -2.3562, True),
)

LOGO_PIECES = (
    ((-0.5000, 0.4146), (-0.1208, 0.3104), (-0.1167, 0.3063), (0.0208, -0.0771)),
    ((0.3625, 0.2979), (0.3625, 0.3146), (0.4958, 0.2771), (0.5000, 0.2729), (0.4042, 0.1854), (0.3958, 0.1812),
     (0.3917, 0.1812)),
    ((0.1833, 0.1521), (0.3292, 0.3021), (0.4000, 0.0104), (0.4000, 0.0063)),
    ((0.3833, -0.0229), (0.3833, -0.0271), (0.0792, -0.1146), (0.0708, -0.1146), (0.0667, -0.1104), (-0.0708, 0.2812)),
    ((0.0750, -0.1521), (0.0792, -0.1479), (0.3667, -0.0646), (0.3792, -0.0646), (0.0833, -0.4146)),
    ((0.0458, -0.1771), (0.0458, -0.3521), (-0.2083, -0.3563), (0.0375, -0.1729), (0.0417, -0.1729)),
)
LOGO_CENTER, LOGO_WIDTH, LOGO_Z, LOGO_INK = (-0.06, 0.0), 0.062, 0.044, 0.0003


def logo_xml():
    meshes, geoms = [], []
    for i, piece in enumerate(LOGO_PIECES):
        verts = []
        for px, py in piece:
            x, y = LOGO_CENTER[0] + py * LOGO_WIDTH, LOGO_CENTER[1] - px * LOGO_WIDTH
            verts += [f"{x:.5f} {y:.5f} {LOGO_Z}", f"{x:.5f} {y:.5f} {LOGO_Z + LOGO_INK:.4f}"]
        meshes.append(f'    <mesh name="logo{i}" vertex="{"  ".join(verts)}"/>')
        geoms.append(f'      <geom class="visual" type="mesh" mesh="logo{i}" material="logo_ink"/>')
    return "\n".join(meshes), "\n".join(geoms)

COXA_STATOR = ('      <geom class="visual" type="box" size="0.015 0.0115 0.0105" pos="{x:.4f} {y:.4f} 0" '
               'euler="0 0 {yaw}" material="servo"/>')

LEG_TOP = """      <body name="{n}_coxa" pos="{x} {y} 0" euler="0 0 {yaw}">
        <joint name="{n}_coxa" class="coxa"/>
        <geom class="coxa" fromto="0 0 0 0.04 0 0"/>
        <geom class="visual" type="cylinder" size="0.0085 0.0012" pos="0 0 0.0117" material="silver"/>
        <geom class="visual" type="cylinder" size="0.0085 0.0012" pos="0 0 -0.0117" material="silver"/>
        <geom class="visual" type="box" size="0.032 0.0125 0.0018" pos="0.028 0 0.0147" material="bracket"/>
        <geom class="visual" type="box" size="0.032 0.0125 0.0018" pos="0.028 0 -0.0147" material="bracket"/>
        <geom class="visual" type="cylinder" size="0.0016 0.0005" pos="0.048 0.006 0.017" material="silver"/>
        <geom class="visual" type="cylinder" size="0.0016 0.0005" pos="0.048 -0.006 0.017" material="silver"/>
        <geom class="visual" type="box" size="0.016 0.0115 0.0105" pos="0.05 0 0" material="servo"/>
        <body name="{n}_femur" pos="0.04 0 0">
          <joint name="{n}_femur" class="femur"/>
          <geom class="femur" fromto="0 0 0 0.08 0 0"/>
          <geom class="visual" type="cylinder" fromto="0 0.0115 0 0 0.0139 0" size="0.0085" material="silver"/>
          <geom class="visual" type="cylinder" fromto="0 -0.0115 0 0 -0.0139 0" size="0.0085" material="silver"/>
          <geom class="visual" type="box" size="0.04 0.0012 0.0095" pos="0.04 0.0151 0" material="bracket"/>
          <geom class="visual" type="box" size="0.04 0.0012 0.0095" pos="0.04 -0.0151 0" material="bracket"/>
          <geom class="visual" type="cylinder" fromto="0 0.0139 0 0 0.0163 0" size="0.0095" material="bracket"/>
          <geom class="visual" type="cylinder" fromto="0 -0.0139 0 0 -0.0163 0" size="0.0095" material="bracket"/>
          <geom class="visual" type="cylinder" fromto="0.08 0.0139 0 0.08 0.0163 0" size="0.0095" material="bracket"/>
          <geom class="visual" type="cylinder" fromto="0.08 -0.0139 0 0.08 -0.0163 0" size="0.0095" material="bracket"/>
          <geom class="visual" type="box" size="0.016 0.0115 0.0105" pos="0.07 0 0" material="servo"/>
          <geom class="visual" type="capsule" fromto="0.064 0.004 0.0112 0.028 0.004 0.0112" size="0.0015" material="cable"/>
          <geom class="visual" type="capsule" fromto="0.028 0.004 0.0112 0.012 0.006 0.0135" size="0.0015" material="cable"/>
          <body name="{n}_tibia" pos="0.08 0 0">
            <joint name="{n}_tibia" class="tibia"/>
            <geom class="tibia" fromto="0 0 0 0 0 {tibia_end}"/>
            <geom class="visual" type="cylinder" fromto="0 0.0115 0 0 0.0139 0" size="0.0085" material="silver"/>
            <geom class="visual" type="cylinder" fromto="0 -0.0115 0 0 -0.0139 0" size="0.0085" material="silver"/>
            <geom class="visual" type="box" size="0.009 0.0012 0.021" pos="0 0.0177 -0.021" material="bracket"/>
            <geom class="visual" type="box" size="0.009 0.0012 0.021" pos="0 -0.0177 -0.021" material="bracket"/>
            <geom class="visual" type="cylinder" fromto="0 0.0165 0 0 0.0189 0" size="0.009" material="bracket"/>
            <geom class="visual" type="cylinder" fromto="0 -0.0165 0 0 -0.0189 0" size="0.009" material="bracket"/>
            <geom class="visual" type="box" size="0.009 0.0189 0.0045" pos="0 0 -0.0445" material="bracket"/>
            <geom class="visual" type="cylinder" size="0.0016 0.0005" pos="0.0092 0.01 -0.0445" euler="0 1.5708 0" material="silver"/>
            <geom class="visual" type="cylinder" size="0.0016 0.0005" pos="0.0092 -0.01 -0.0445" euler="0 1.5708 0" material="silver"/>
"""

FOOT_END = """            <geom class="visual" type="capsule" fromto="0 0 -0.049 0 0 -0.108" size="0.0058" material="carbon"/>
            <geom class="visual" type="cylinder" size="0.0082 0.003" pos="0 0 -0.106" material="rubber"/>
            <geom name="{n}_foot" class="foot" pos="0 0 -0.12"/>
          </body>
        </body>
      </body>"""

WHEEL_END = """            <geom class="visual" type="capsule" fromto="0 0 -0.049 0 0 -0.094" size="0.0058" material="carbon"/>
            <geom class="visual" type="box" size="0.009 0.009 0.009" pos="0 0 -{drop}" material="bracket"/>
            <geom class="visual" type="cylinder" fromto="0 0 -{drop} {stub}" size="0.0045" material="silver"/>
            <body name="{n}_wheel" pos="{hub} -{drop}">
              <joint name="{n}_wheel" class="wheel" axis="{axis} 0"/>
              <geom name="{n}_tire" class="tire"/>
              <geom class="visual" type="cylinder" fromto="{tread}" size="{radius}" material="rubber"/>
              <geom class="visual" type="cylinder" fromto="{rim}" size="0.0195" material="bracket"/>
              <geom class="visual" type="cylinder" fromto="{face}" size="0.014" material="servo"/>
{bolts}
            </body>
          </body>
        </body>
      </body>"""


def blur(a, cells):
    x = np.arange(-3 * cells, 3 * cells + 1)
    k = np.exp(-0.5 * (x / cells) ** 2)
    k /= k.sum()
    for axis in (0, 1):
        a = np.apply_along_axis(lambda r: np.convolve(r, k, mode="same"), axis, a)
    return a


def tile_heights(kind, d, rng, n):
    u = (np.arange(n + 1) * RES)[None, :]
    v = (np.arange(n + 1) * RES)[:, None]
    shape = (n + 1, n + 1)
    cheb = np.maximum(np.abs(u - TILE / 2), np.abs(v - TILE / 2))
    plateau = np.clip(0.8 - cheb, 0.0, 0.6)

    def bumps(amp, scale):
        b = blur(rng.standard_normal(shape), scale / RES)
        return amp * (b - b.min()) / (b.max() - b.min())

    def stones(amp, size):
        k = max(1, round(size / RES))
        m = -(-(n + 1) // k)
        return amp * np.kron(rng.uniform(0.0, 1.0, (m, m)), np.ones((k, k)))[: n + 1, : n + 1]

    def blocks(count, size, height, clear=0.3):
        h = np.zeros(shape)
        placed = 0
        while placed < count:
            cx, cy = rng.uniform(0.2, TILE - 0.2, 2)
            if max(abs(cx - TILE / 2), abs(cy - TILE / 2)) < clear:
                continue
            sx, sy = rng.uniform(size[0], size[1], 2) / 2
            top = rng.uniform(0.5, 1.0) * height
            h = np.where((np.abs(u - cx) < sx) & (np.abs(v - cy) < sy), np.maximum(h, top), h)
            placed += 1
        return h

    if kind == "gravel":
        h = stones(0.003 + 0.017 * d, 0.04)
    elif kind == "ramp":
        h = math.tan(math.radians(4 + 22 * d)) * plateau
    elif kind == "bumps":
        h = bumps(0.01 + 0.07 * d, 0.12)
    elif kind == "rubble":
        h = stones(0.005 + 0.035 * d, 0.08) + stones(0.003 + 0.007 * d, 0.04)
    elif kind == "blocks":
        h = blocks(int(10 + 16 * d), (0.1, 0.3), 0.015 + 0.065 * d)
    elif kind == "stairs":
        h = np.floor(plateau / 0.2 + 1e-9) * (0.015 + 0.055 * d)
    elif kind == "trenches":
        width = 0.03 + 0.09 * d
        gap = (np.abs(cheb - 0.35) < width / 2) | (np.abs(cheb - 0.65) < width / 2)
        h = (0.04 + 0.03 * d) * ~gap
    else:
        h = (math.tan(math.radians(3 + 12 * d)) * plateau + stones(0.002 + 0.018 * d, 0.06)
             + blocks(int(4 + 6 * d), (0.1, 0.2), 0.01 + 0.04 * d))
    w = np.clip(np.minimum(np.minimum(u, TILE - u), np.minimum(v, TILE - v)) / MARGIN, 0.0, 1.0)
    return h * w * w * (3 - 2 * w)


def write_png(path, rgb):
    h, w, _ = rgb.shape
    raw = b"".join(b"\x00" + rgb[y].tobytes() for y in range(h))
    chunk = lambda tag, data: (struct.pack(">I", len(data)) + tag + data
                               + struct.pack(">I", zlib.crc32(tag + data) & 0xFFFFFFFF))
    path.write_bytes(b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
                     + chunk(b"IDAT", zlib.compress(raw, 6)) + chunk(b"IEND", b""))


def terrain_texture(heights, up=2):
    h = np.kron(heights, np.ones((up, up)))
    px = RES / up
    gy, gx = np.gradient(h, px)
    normal = np.stack([-gx, -gy, np.ones_like(h)], -1)
    normal /= np.linalg.norm(normal, axis=-1, keepdims=True)
    light = np.array([-0.6, 0.5, 0.75])
    shade = np.clip(normal @ (light / np.linalg.norm(light)), 0.0, 1.0)
    tint = 0.9 + 0.35 * np.clip(h / max(heights.max(), 1e-6), 0.0, 1.0)
    rgb = np.array([0.62, 0.59, 0.54]) * ((0.25 + 0.85 * shade) * tint)[..., None]
    rows, cols = np.indices(h.shape)
    x, y = cols * px - BORDER, rows * px - BORDER
    fine = ((x + 1e-9) % 0.5 < px) | ((y + 1e-9) % 0.5 < px)
    tile = ((x + 1e-9) % TILE < 1.5 * px) | ((y + 1e-9) % TILE < 1.5 * px)
    rgb = np.where(fine[..., None], rgb * 0.82, rgb)
    rgb = np.where(tile[..., None], rgb * 0.65, rgb)
    return (np.clip(rgb, 0.0, 1.0) * 255).astype(np.uint8)


def make_terrain(seed=0):
    rng = np.random.default_rng(seed)
    n, b = round(TILE / RES), round(BORDER / RES)
    heights = np.zeros((len(TERRAIN_TYPES) * n + 1 + 2 * b, LEVELS * n + 1 + 2 * b))
    for j, kind in enumerate(TERRAIN_TYPES):
        for i in range(LEVELS):
            rows, cols = slice(b + j * n, b + (j + 1) * n + 1), slice(b + i * n, b + (i + 1) * n + 1)
            heights[rows, cols] = np.maximum(heights[rows, cols], tile_heights(kind, i / (LEVELS - 1), rng, n))
    for path, data in ((TERRAIN_FILE, heights), (TERRAIN_LITE_FILE, heights[::2, ::2])):
        with open(path, "wb") as f:
            np.array(data.shape, np.int32).tofile(f)
            data.astype(np.float32).tofile(f)
    return heights


def vec(*parts):
    return " ".join(f"{v:.5f}" for p in parts for v in p)


def wheel_end(name, yaw):
    axle = (math.sin(yaw), math.cos(yaw), 0.0)
    ahead = (math.cos(yaw), -math.sin(yaw), 0.0)
    side = 1.0 if name[0] == "L" else -1.0
    along = lambda d: tuple(side * d * a for a in axle)
    span = lambda d: vec(tuple(-d * a for a in axle), tuple(d * a for a in axle))
    bolts = []
    for k in range(5):
        th = 2 * math.pi * k / 5
        off = tuple(0.0095 * (math.cos(th) * h + math.sin(th) * z) for h, z in zip(ahead, (0.0, 0.0, 1.0)))
        start = tuple(o + a for o, a in zip(off, along(0.0136)))
        end = tuple(o + a for o, a in zip(off, along(0.0143)))
        bolts.append(f'              <geom class="visual" type="cylinder" fromto="{vec(start, end)}" size="0.0016" material="silver"/>')
    hub = along(WHEEL_OFFSET)
    return WHEEL_END.format(
        n=name, drop=WHEEL_DROP, radius=WHEEL_RADIUS,
        stub=vec(tuple(c for c in along(WHEEL_OFFSET - 0.011)[:2]) + (-WHEEL_DROP,)),
        hub=vec(hub[:2]), axis=vec(axle[:2]),
        tread=span(0.011), rim=span(0.0116),
        face=vec(along(0.0116), along(0.0136)), bolts="\n".join(bolts))


HEAD = """<mujoco model="hexapod">
  <!-- Generated by hexapod/model.py; do not edit by hand. -->
  <compiler angle="radian" autolimits="true"/>
  <option timestep="0.004" iterations="6" ls_iterations="12"/>

  <visual>
    <headlight ambient="0.25 0.25 0.25" diffuse="0.25 0.25 0.25" specular="0.05 0.05 0.05"/>
    <quality shadowsize="4096" offsamples="8"/>
    <global offwidth="1280" offheight="720" azimuth="145" elevation="-16"/>
    <rgba haze="0.44 0.45 0.46 1"/>
  </visual>

  <asset>
    <texture name="sky" type="skybox" builtin="gradient" rgb1="0.6 0.61 0.63" rgb2="0.33 0.34 0.35" width="512" height="512"/>
    <texture name="floor" type="2d" builtin="checker" rgb1="0.43 0.43 0.44" rgb2="0.45 0.45 0.46" mark="edge" markrgb="0.37 0.37 0.38" width="512" height="512"/>
    <material name="floor" texture="floor" texrepeat="1 1" texuniform="true"/>
    <texture name="terrain" type="2d" file="terrain_tex.png"/>
    <material name="terrain" texture="terrain" texrepeat="1 1"/>
    <hfield name="terrain" file="terrain.bin" size="{terrain_size}"/>
    <hfield name="terrain_lite" file="terrain_lite.bin" size="{terrain_lite_size}"/>
    <material name="carbon" rgba="0.06 0.06 0.065 1" specular="0.6" shininess="0.85"/>
    <material name="servo" rgba="0.08 0.08 0.085 1" specular="0.15" shininess="0.25"/>
    <material name="bracket" rgba="0.5 0.51 0.53 1" specular="0.7" shininess="0.6"/>
    <material name="silver" rgba="0.66 0.67 0.69 1" specular="0.8" shininess="0.7"/>
    <material name="rubber" rgba="0.13 0.13 0.13 1" specular="0.03" shininess="0.05"/>
    <material name="pcb" rgba="0.05 0.22 0.12 1" specular="0.35" shininess="0.5"/>
    <material name="chip" rgba="0.03 0.03 0.035 1" specular="0.4" shininess="0.5"/>
    <material name="battery" rgba="0.14 0.15 0.17 1" specular="0.55" shininess="0.65"/>
    <material name="camera" rgba="0.2 0.2 0.21 1" specular="0.45" shininess="0.55"/>
    <material name="glass" rgba="0.015 0.015 0.02 1" specular="1" shininess="1" reflectance="0.25"/>
    <material name="lens" rgba="0.02 0.02 0.025 1" specular="1" shininess="1"/>
    <material name="lidar" rgba="0.12 0.12 0.13 1" specular="0.4" shininess="0.5"/>
    <material name="cable" rgba="0.03 0.03 0.03 1" specular="0.2" shininess="0.3"/>
    <material name="led" rgba="0.25 1 0.35 1" emission="0.8"/>
    <material name="shell" rgba="0.1 0.1 0.105 1" specular="0.35" shininess="0.5"/>
    <material name="logo_ink" rgba="0.93 0.93 0.94 1" specular="0.25" shininess="0.3"/>
    <mesh name="plate" vertex="0.17 0 -0.003  0.12 0.095 -0.003  -0.12 0.095 -0.003  -0.17 0 -0.003  -0.12 -0.095 -0.003  0.12 -0.095 -0.003
                               0.17 0 0.003  0.12 0.095 0.003  -0.12 0.095 0.003  -0.17 0 0.003  -0.12 -0.095 0.003  0.12 -0.095 0.003"/>
{logo_meshes}
  </asset>

  <default>
    <joint damping="0.1" armature="0.002"/>
    <geom contype="0" conaffinity="0"/>
    <motor ctrlrange="-2 2"/>
    <default class="coxa">
      <joint axis="0 0 1" range="-0.6 0.6"/>
      <geom type="capsule" size="0.012" mass="0.04" group="3" rgba="0 0 0 0"/>
    </default>
    <default class="femur">
      <joint axis="0 1 0" range="-0.9 0.9"/>
      <geom type="capsule" size="0.01" mass="0.06" group="3" rgba="0 0 0 0"/>
    </default>
    <default class="tibia">
      <joint axis="0 1 0" range="-0.9 0.9"/>
      <geom type="capsule" size="0.008" mass="0.03" group="3" rgba="0 0 0 0"/>
    </default>
    <default class="foot">
      <geom type="sphere" size="0.012" mass="0.01" material="rubber" conaffinity="3" condim="3" friction="1 0.005 0.0001"/>
    </default>
    <default class="wheel">
      <joint damping="0.005" armature="0.0005"/>
    </default>
    <default class="tire">
      <geom type="sphere" size="{radius}" mass="0.07" group="3" rgba="0 0 0 0" conaffinity="3" condim="3" friction="1 0.005 0.0001"/>
    </default>
    <default class="belly">
      <geom type="sphere" size="0.025" mass="0" group="3" rgba="0 0 0 0" conaffinity="3" condim="3" friction="0.6 0.005 0.0001"/>
    </default>
    <default class="visual">
      <geom density="0"/>
    </default>
  </default>

  <worldbody>
    <light name="fill" directional="true" pos="-1 1 3" dir="0.4 -0.4 -1" diffuse="0.35 0.35 0.36" specular="0.05 0.05 0.05" castshadow="false"/>
    <light name="rim" directional="true" pos="-1 1 1.5" dir="0.5 -0.5 -0.5" diffuse="0.22 0.22 0.23" specular="0.35 0.35 0.35" castshadow="false"/>
    <geom name="floor" type="plane" size="0 0 0.05" pos="0 0 -0.005" material="floor" contype="1" conaffinity="0" condim="3" friction="1 0.005 0.0001"/>
    <geom name="terrain" type="hfield" hfield="terrain" pos="{terrain_pos}" material="terrain" group="2" contype="2" conaffinity="0" condim="3" friction="1 0.005 0.0001"/>
    <geom name="terrain_lite" type="hfield" hfield="terrain_lite" pos="{terrain_pos}" material="terrain" group="4" contype="0" conaffinity="0"/>
    <body name="torso" pos="0 0 0.145">
      <freejoint name="root"/>
      <light name="key" mode="trackcom" pos="0.8 -1.1 2" dir="-0.35 0.5 -1" diffuse="0.9 0.9 0.9" specular="0.4 0.4 0.4" cutoff="70" exponent="2" castshadow="true"/>
      <geom name="torso" type="box" size="0.15 0.08 0.025" mass="1.2" group="3" rgba="0 0 0 0"/>
      <geom name="belly_front" class="belly" pos="0.09 0 -0.002"/>
      <geom name="belly_rear" class="belly" pos="-0.09 0 -0.002"/>
      <geom name="head" type="sphere" pos="0.16 0 0.01" size="0.025" mass="0.05" group="3" rgba="0 0 0 0"/>

      <geom class="visual" type="mesh" mesh="plate" pos="0 0 0.022" material="carbon"/>
      <geom class="visual" type="mesh" mesh="plate" pos="0 0 -0.022" material="carbon"/>
      <geom class="visual" type="cylinder" size="0.004 0.019" pos="0.06 0.085 0" material="silver"/>
      <geom class="visual" type="cylinder" size="0.004 0.019" pos="0.06 -0.085 0" material="silver"/>
      <geom class="visual" type="cylinder" size="0.004 0.019" pos="-0.06 0.085 0" material="silver"/>
      <geom class="visual" type="cylinder" size="0.004 0.019" pos="-0.06 -0.085 0" material="silver"/>
      <geom class="visual" type="cylinder" size="0.004 0.019" pos="-0.14 0.03 0" material="silver"/>
      <geom class="visual" type="cylinder" size="0.004 0.019" pos="-0.14 -0.03 0" material="silver"/>
      <geom class="visual" type="cylinder" size="0.0028 0.0006" pos="0.06 0.085 0.0256" material="silver"/>
      <geom class="visual" type="cylinder" size="0.0028 0.0006" pos="0.06 -0.085 0.0256" material="silver"/>
      <geom class="visual" type="cylinder" size="0.0028 0.0006" pos="-0.06 0.085 0.0256" material="silver"/>
      <geom class="visual" type="cylinder" size="0.0028 0.0006" pos="-0.06 -0.085 0.0256" material="silver"/>
      <geom class="visual" type="cylinder" size="0.0028 0.0006" pos="-0.14 0.03 0.0256" material="silver"/>
      <geom class="visual" type="cylinder" size="0.0028 0.0006" pos="-0.14 -0.03 0.0256" material="silver"/>
      <geom class="visual" type="box" size="0.05 0.03 0.011" pos="-0.045 0 -0.006" material="battery"/>
      <geom class="visual" type="box" size="0.004 0.0303 0.0113" pos="-0.03 0 -0.006" material="cable"/>
      <geom class="visual" type="box" size="0.03 0.028 0.0008" pos="0.045 0 -0.0172" material="pcb"/>
      <geom class="visual" type="box" size="0.007 0.007 0.001" pos="0.045 0.006 -0.0154" material="chip"/>
      <geom class="visual" type="box" size="0.004 0.009 0.003" pos="0.062 -0.012 -0.0134" material="chip"/>

      <geom class="visual" type="cylinder" size="0.0015 0.003" pos="-0.025 0.024 0.028" material="silver"/>
      <geom class="visual" type="cylinder" size="0.0015 0.003" pos="-0.025 -0.024 0.028" material="silver"/>
      <geom class="visual" type="cylinder" size="0.0015 0.003" pos="-0.095 0.024 0.028" material="silver"/>
      <geom class="visual" type="cylinder" size="0.0015 0.003" pos="-0.095 -0.024 0.028" material="silver"/>
      <geom class="visual" type="box" size="0.042 0.029 0.0008" pos="-0.06 0 0.0318" material="pcb"/>
      <geom class="visual" type="box" size="0.007 0.007 0.0008" pos="-0.052 0.004 0.0334" material="chip"/>
      <geom class="visual" type="box" size="0.006 0.0035 0.0007" pos="-0.038 -0.01 0.0333" material="chip"/>
      <geom class="visual" type="box" size="0.022 0.0025 0.003" pos="-0.06 0.0255 0.0356" material="chip"/>
      <geom class="visual" type="box" size="0.007 0.0065 0.0045" pos="-0.097 0.011 0.0371" material="silver"/>
      <geom class="visual" type="box" size="0.007 0.0065 0.0045" pos="-0.097 -0.008 0.0371" material="silver"/>
      <geom class="visual" type="box" size="0.024 0.02 0.004" pos="0.055 0 0.029" material="servo"/>
      <geom class="visual" type="cylinder" size="0.017 0.009" pos="0.06 0 0.042" material="lidar"/>
      <geom class="visual" type="cylinder" size="0.0172 0.0028" pos="0.06 0 0.043" material="lens"/>
      <geom class="visual" type="cylinder" size="0.0165 0.0012" pos="0.06 0 0.0512" material="servo"/>
      <geom class="visual" type="box" size="0.005 0.008 0.0022" pos="0.15 0 0.0272" material="bracket"/>
      <geom class="visual" type="box" size="0.0085 0.045 0.0105" pos="0.15 0 0.0399" material="camera"/>
      <geom class="visual" type="box" size="0.0005 0.043 0.0085" pos="0.1588 0 0.0399" material="glass"/>
      <geom class="visual" type="cylinder" fromto="0.1588 -0.031 0.0399 0.1594 -0.031 0.0399" size="0.0052" material="silver"/>
      <geom class="visual" type="cylinder" fromto="0.1588 -0.012 0.0399 0.1594 -0.012 0.0399" size="0.0034" material="silver"/>
      <geom class="visual" type="cylinder" fromto="0.1588 0.007 0.0399 0.1594 0.007 0.0399" size="0.0052" material="silver"/>
      <geom class="visual" type="cylinder" fromto="0.1588 0.03 0.0399 0.1594 0.03 0.0399" size="0.0052" material="silver"/>
      <geom class="visual" type="cylinder" fromto="0.1594 -0.031 0.0399 0.1599 -0.031 0.0399" size="0.0042" material="lens"/>
      <geom class="visual" type="cylinder" fromto="0.1594 -0.012 0.0399 0.1599 -0.012 0.0399" size="0.0026" material="lens"/>
      <geom class="visual" type="cylinder" fromto="0.1594 0.007 0.0399 0.1599 0.007 0.0399" size="0.0042" material="lens"/>
      <geom class="visual" type="cylinder" fromto="0.1594 0.03 0.0399 0.1599 0.03 0.0399" size="0.0042" material="lens"/>
      <geom class="visual" type="sphere" size="0.0016" pos="-0.13 0.035 0.0262" material="led"/>
      <geom class="visual" type="box" size="0.047 0.036 0.0095" pos="-0.06 0 0.0345" material="shell"/>
{logo}

{stators}

"""

TAIL = """    </body>
  </worldbody>

  <actuator>
{motors}
  </actuator>
</mujoco>
"""


def main():
    ASSETS.mkdir(exist_ok=True)
    stators = "\n".join(COXA_STATOR.format(x=x - 0.010 * math.cos(yaw), y=y - 0.010 * math.sin(yaw), yaw=yaw)
                        for _, (x, y), yaw, _ in LEGS)
    legs = []
    for name, (x, y), yaw, wheel in LEGS:
        top = LEG_TOP.format(n=name, x=x, y=y, yaw=yaw, tibia_end=-0.09 if wheel else -0.105)
        legs.append(top + (wheel_end(name, yaw) if wheel else FOOT_END.format(n=name)))
    motors = [f'    <motor name="{n}_{s}" joint="{n}_{s}"/>' for n, *_ in LEGS for s in ("coxa", "femur", "tibia")]
    motors += [f'    <motor name="{n}_wheel" joint="{n}_wheel" ctrlrange="-0.6 0.6"/>' for n, *_, wheel in LEGS if wheel]
    heights = make_terrain()
    write_png(TEXTURE_FILE, terrain_texture(heights)[::-1])
    half_x, half_y = LEVELS * TILE / 2 + BORDER, len(TERRAIN_TYPES) * TILE / 2 + BORDER
    logo_meshes, logo_geoms = logo_xml()
    head = (HEAD.replace("{radius}", str(WHEEL_RADIUS)).replace("{stators}", stators)
            .replace("{logo_meshes}", logo_meshes).replace("{logo}", logo_geoms)
            .replace("{terrain_size}", f"{half_x} {half_y} {heights.max():.6f} 0.1")
            .replace("{terrain_lite_size}", f"{half_x} {half_y} {heights[::2, ::2].max():.6f} 0.1")
            .replace("{terrain_pos}", f"{START_X - BORDER + half_x} 0 0"))
    OUT.write_text(head + "\n".join(legs) + "\n" + TAIL.format(motors="\n".join(motors)), encoding="utf-8")
    digest = hashlib.sha1(TERRAIN_FILE.read_bytes()).hexdigest()[:12]
    print(f"wrote {OUT.name} and {TERRAIN_FILE.name}; terrain {2 * half_x:.0f} x {2 * half_y:.0f} m, "
          f"{heights.shape[1]}x{heights.shape[0]} cells, highest point {heights.max() * 100:.1f} cm, digest {digest}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile hexapod/env.py
"""CPU MuJoCo environment: viewing, evaluation and video."""
from collections import deque

import mujoco
import numpy as np

from . import core


class HexapodEnv:
    def __init__(self, failures=True, seed=None):
        self.model = mujoco.MjModel.from_xml_path(str(core.MODEL_PATH))
        self.data = mujoco.MjData(self.model)
        self.failures = failures
        self.rng = np.random.default_rng(seed)
        self.qadr, self.vadr, self.wadr, self.contact, self.contact_r = core.addresses(self.model)
        self.terrain = core.load_terrain(self.model)
        self.external_cmd = None
        self.level = self.column = None
        self._renderer = None

    def reset(self, seed=None, level=None, column=None, yaw=None, pos=(0.0, 0.0)):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        if isinstance(column, str):
            column = core.TERRAIN_TYPES.index(column)
        self.level, self.column = level, column
        rng, d = self.rng, self.data
        mujoco.mj_resetData(self.model, d)
        if level is None:
            x, y = pos
            yaw = 0.0 if yaw is None else yaw
        else:
            cx, cy = core.tile_center(level, column)
            x, y = cx + rng.uniform(-core.SPAWN_NOISE, core.SPAWN_NOISE), cy + rng.uniform(-core.SPAWN_NOISE, core.SPAWN_NOISE)
            yaw = rng.uniform(-np.pi, np.pi) if yaw is None else yaw
        d.qpos[0:3] = x, y, core.spawn_height(np, self.terrain, x, y)
        d.qpos[3:7] = np.cos(yaw / 2), 0.0, 0.0, np.sin(yaw / 2)
        d.qpos[self.qadr] += rng.uniform(-0.05, 0.05, core.N_JOINTS)
        mujoco.mj_forward(self.model, d)
        self.t = 0
        self.last_action = np.zeros(core.N_ACT)
        self.cmd_seq, self.cmd_t = self._sample_commands(level or 0)
        if self.failures:
            self.mode, self.strength, self.fail_t = self._sample_failures()
        else:
            self.mode = np.full(core.N_ACT, core.HEALTHY)
            self.strength = np.ones(core.N_ACT)
            self.fail_t = np.full(core.N_ACT, core.NEVER)
        self.q_lock = d.qpos[self.qadr].copy()
        self.air_time = np.zeros(core.N_LEGS)
        self.last_contact = np.ones(core.N_LEGS, bool)
        self.phase = 0.0
        self.history = deque([self._actor_obs()] * core.HISTORY, maxlen=core.HISTORY)
        scan = core.scan_heights(np, self.terrain, d.qpos)
        return self._observation(scan), {}

    @property
    def cmd(self):
        if self.external_cmd is not None:
            return self.external_cmd
        return core.command_at(np, self.cmd_seq, self.cmd_t, self.t)

    def set_command(self, cmd):
        self.external_cmd = None if cmd is None else np.asarray(cmd, dtype=np.float64)

    def fail_leg(self, leg, mode=core.LIMP, strength=1.0):
        self.fail_actuators(np.flatnonzero(core.ACT_LEG == leg), mode, strength)

    def fail_wheels(self, mode=core.LOCKED):
        self.fail_actuators(np.flatnonzero(core.IS_WHEEL), mode)

    def fail_actuators(self, idx, mode=core.LIMP, strength=1.0):
        self.mode[idx], self.strength[idx], self.fail_t[idx] = mode, strength, self.t

    def step(self, action):
        action = np.clip(np.asarray(action, dtype=np.float64), -1.0, 1.0)
        cmd = self.cmd
        action = core.park_brake(np, action, cmd)
        newly = self.fail_t[:core.N_JOINTS] == self.t
        self.q_lock[newly] = self.data.qpos[self.qadr][newly]
        mode = core.active_modes(np, self.mode, self.fail_t, self.t)
        d = self.data
        for _ in range(core.N_SUBSTEPS):
            tau = core.actuator_torque(np, action, d.qpos[self.qadr], d.qvel[self.vadr], d.qvel[self.wadr],
                                       mode, self.strength, self.q_lock)
            d.ctrl[:] = tau
            mujoco.mj_step(self.model, d)
        scan = core.scan_heights(np, self.terrain, d.qpos)
        ground = core.ground_level(np, scan)
        normal = core.ground_normal(np, self.terrain, d.qpos[0], d.qpos[1])
        gaps = core.foot_gaps(np, self.terrain, d.geom_xpos[self.contact], self.contact_r)
        contact = gaps < core.CONTACT_TOL
        self.air_time, landed, first = core.air_time_step(np, self.air_time, self.last_contact, contact)
        self.last_contact = contact
        self.phase = float(core.gait_step(np, self.phase, cmd))
        terms = core.reward_terms(np, d.qpos, d.qvel, self.qadr, self.vadr, self.wadr, cmd, action,
                                  self.last_action, tau, mode, ground, normal, contact, landed, first,
                                  self.phase, gaps)
        self.last_action = action
        self.t += 1
        self.history.append(self._actor_obs())
        obs = self._observation(scan)
        terminated = bool(core.fallen(np, d.qpos, ground)) or not np.isfinite(obs).all()
        info = {
            "velocity": core.body_velocity(np, d.qpos, d.qvel),
            "cmd": cmd,
            "failed": int((mode != core.HEALTHY).sum()),
            "ground": float(ground),
            "contact": contact,
            **{f"r_{k}": float(v) for k, v in terms.items()},
        }
        return obs, float(sum(terms.values())), terminated, self.t >= core.EPISODE_STEPS, info

    def render(self, width=640, height=480, distance=0.8, elevation=-16.0, azimuth=145.0):
        if self._renderer is None:
            self._renderer = mujoco.Renderer(self.model, height, width)
            self._camera = mujoco.MjvCamera()
            self._camera.type = mujoco.mjtCamera.mjCAMERA_TRACKING
            self._camera.trackbodyid = self.model.body("torso").id
        self._camera.distance, self._camera.elevation, self._camera.azimuth = distance, elevation, azimuth
        self._renderer.update_scene(self.data, self._camera)
        return self._renderer.render()

    def close(self):
        if self._renderer is not None:
            self._renderer.close()
            self._renderer = None

    def _actor_obs(self):
        d = self.data
        return core.actor_obs(np, d.qpos, d.qvel, self.qadr, self.vadr, self.wadr,
                              self.last_action, self.cmd, self.phase).astype(np.float32)

    def _observation(self, scan):
        return np.concatenate([*self.history, core.scan_obs(np, self.data.qpos, scan).astype(np.float32)])

    def _sample_commands(self, level):
        holds = self.rng.integers(*core.CMD_HOLD, core.CMD_SEGMENTS)
        cmd_t = np.concatenate([[0], np.cumsum(holds)[:-1]])
        high = core.CMD_HIGH.copy()
        high[0] -= core.SPEED_DROP * level
        cmd = core.CMD_LOW + self.rng.random((core.CMD_SEGMENTS, 3)) * (high - core.CMD_LOW)
        cmd[self.rng.random(core.CMD_SEGMENTS) < core.P_STAND] = 0.0
        return cmd, cmd_t

    def _sample_failures(self):
        rng = self.rng
        mode = np.full(core.N_ACT, core.HEALTHY)
        strength = np.ones(core.N_ACT)
        fail_t = np.full(core.N_ACT, core.NEVER)
        kind = min(int(np.searchsorted(core.FAIL_CUM, rng.random(), side="right")), len(core.FAIL_P) - 1)
        legs = rng.permutation(core.N_LEGS)
        groups = []
        if kind == core.ONE_MOTOR:
            groups = [np.array([rng.integers(core.N_ACT)])]
        elif kind in (core.ONE_LEG, core.WHEELS_AND_LEG):
            groups = [np.flatnonzero(core.ACT_LEG == legs[0])]
        elif kind == core.TWO_LEGS:
            groups = [np.flatnonzero(core.ACT_LEG == legs[0]), np.flatnonzero(core.ACT_LEG == legs[1])]
        when = lambda: 0 if rng.random() < 0.5 else rng.integers(core.FAIL_EARLIEST, core.FAIL_LATEST)
        for idx in groups:
            v = rng.random()
            kind_ = core.LIMP if v < core.P_LIMP else core.LOCKED if v < core.P_LIMP + core.P_LOCKED else core.WEAK
            mode[idx] = kind_
            strength[idx] = rng.uniform(*core.WEAK_RANGE) if kind_ == core.WEAK else 1.0
            fail_t[idx] = when()
        if kind >= core.ALL_WHEELS:
            wheels = np.flatnonzero(core.IS_WHEEL)
            mode[wheels] = core.LIMP if rng.random() < core.P_WHEELS_LIMP else core.LOCKED
            strength[wheels] = 1.0
            fail_t[wheels] = when()
        return mode, strength, fail_t

In [ ]:
%%writefile hexapod/mjx_env.py
"""MJX (GPU) environment with terrain curriculum, used for training."""
import jax
import jax.numpy as jnp
import mujoco
import numpy as np
from brax.envs.base import Env, State, Wrapper
from brax.envs.wrappers.training import EpisodeWrapper, VmapWrapper
from mujoco import mjx
from mujoco.mjx._src import collision_convex, collision_driver
from mujoco.mjx._src import math as mjx_math
from mujoco.mjx._src.types import GeomType

from . import core


@collision_convex.collider(ncon=1)
def _hfield_sphere_single(h, s, subgrid_size):
    dist, pos, n = collision_convex._hfield_collision(collision_convex._sphere_convex, h, s, jnp.max(s.size),
                                                      subgrid_size)
    i = jnp.argmin(dist)
    return dist[i], h.mat @ pos[i] + h.pos, mjx_math.make_frame(h.mat @ n[i])


collision_driver._COLLISION_FUNC[(GeomType.HFIELD, GeomType.SPHERE)] = _hfield_sphere_single

INFO_KEYS = ("rng", "t", "cmd_seq", "cmd_t", "mode", "strength", "fail_t", "q_lock", "last_action",
             "air_time", "last_contact", "level", "column", "spawn_xy", "cmd_disp", "friction", "softness", "phase")
METRIC_KEYS = tuple(f"r_{k}" for k in core.REWARD_KEYS) + (
    "vx", "vy", "wz", "lin_err", "yaw_err", "failed", "wheels_dead", "level_per_step")
CURRICULUM = {"levels": np.full(core.LEVELS, np.nan)}


def _record_levels(histogram):
    CURRICULUM["levels"] = np.asarray(histogram, np.float64)


class HexapodMJX(Env):
    def __init__(self, failures=True, curriculum=True, randomize=None, init_level_max=core.INIT_LEVEL_MAX):
        self.mj_model = mujoco.MjModel.from_xml_path(str(core.MODEL_PATH))
        self.mjx_model = mjx.put_model(self.mj_model)
        self.qadr, self.vadr, self.wadr, self.contact, self.contact_r = core.addresses(self.mj_model)
        terrain = core.load_terrain(self.mj_model)
        self.terrain = terrain._replace(heights=jnp.asarray(terrain.heights, jnp.float32))
        self.failures, self.curriculum = failures, curriculum
        self.randomize = curriculum if randomize is None else randomize
        self.init_level_max = init_level_max

    @property
    def observation_size(self):
        return {"state": (core.OBS_SIZE,), "privileged_state": (core.CRITIC_OBS,)}

    @property
    def action_size(self):
        return core.N_ACT

    @property
    def backend(self):
        return "mjx"

    def reset(self, rng):
        rng, k_level, k_column = jax.random.split(rng, 3)
        top = self.init_level_max if self.curriculum else core.LEVELS - 1
        level = jax.random.randint(k_level, (), 0, top + 1)
        column = jax.random.randint(k_column, (), 0, core.N_TYPES)
        return self.reset_at(rng, level, column)

    def reset_at(self, rng, level, column):
        rng, k_pose, k_xy, k_yaw, k_cmd, k_fail, k_friction, k_soft, k_edge, k_side, k_phase = jax.random.split(rng, 11)
        cx, cy = core.tile_center(level, column)
        edge = jax.random.uniform(k_edge) < core.P_EDGE_SPAWN
        side = jnp.where(jax.random.uniform(k_side) < 0.5, -1.0, 1.0)
        cx = cx + jnp.where(edge, side * core.EDGE_OFFSET, 0.0)
        xy = jnp.stack([cx, cy]) + jax.random.uniform(k_xy, (2,), minval=-core.SPAWN_NOISE, maxval=core.SPAWN_NOISE)
        yaw = jax.random.uniform(k_yaw, minval=-jnp.pi, maxval=jnp.pi)
        z = core.spawn_height(jnp, self.terrain, xy[0], xy[1])
        qpos = jnp.asarray(self.mj_model.qpos0, jnp.float32)
        qpos = qpos.at[0:3].set(jnp.stack([xy[0], xy[1], z]))
        qpos = qpos.at[3:7].set(jnp.stack([jnp.cos(yaw / 2), 0.0, 0.0, jnp.sin(yaw / 2)]))
        qpos = qpos.at[self.qadr].add(jax.random.uniform(k_pose, (core.N_JOINTS,), minval=-0.05, maxval=0.05))
        data = mjx.make_data(self.mj_model).replace(qpos=qpos)
        cmd_seq, cmd_t = self._sample_commands(k_cmd, level)
        if self.failures:
            mode, strength, fail_t = self._sample_failures(k_fail)
        else:
            mode = jnp.full(core.N_ACT, core.HEALTHY, jnp.int32)
            strength = jnp.ones(core.N_ACT)
            fail_t = jnp.full(core.N_ACT, core.NEVER, jnp.int32)
        t = jnp.zeros((), jnp.int32)
        last_action = jnp.zeros(core.N_ACT)
        contact = jnp.ones(core.N_LEGS, bool)
        air_time = jnp.zeros(core.N_LEGS)
        phase = jax.random.uniform(k_phase)
        actor_now = core.actor_obs(jnp, qpos, data.qvel, self.qadr, self.vadr, self.wadr, last_action,
                                   core.command_at(jnp, cmd_seq, cmd_t, t), phase)
        obs = self._obs(jnp.tile(actor_now, core.HISTORY), actor_now, qpos, data.qvel,
                        core.active_modes(jnp, mode, fail_t, t), strength, contact, air_time)
        info = {"rng": rng, "t": t, "cmd_seq": cmd_seq, "cmd_t": cmd_t, "mode": mode, "strength": strength,
                "fail_t": fail_t, "q_lock": qpos[self.qadr], "last_action": last_action, "air_time": air_time,
                "last_contact": contact, "level": jnp.asarray(level, jnp.int32), "column": jnp.asarray(column, jnp.int32),
                "spawn_xy": xy, "cmd_disp": jnp.zeros(2), "phase": phase}
        if self.randomize:
            info["friction"] = jax.random.uniform(k_friction, minval=core.FRICTION_RANGE[0], maxval=core.FRICTION_RANGE[1])
            info["softness"] = jax.random.uniform(k_soft, minval=core.SOFTNESS_RANGE[0], maxval=core.SOFTNESS_RANGE[1])
        else:
            info["friction"], info["softness"] = jnp.ones(()), jnp.full((), 0.02)
        metrics = {key: jnp.zeros(()) for key in METRIC_KEYS}
        return State(pipeline_state=data, obs=obs, reward=jnp.zeros(()), done=jnp.zeros(()), metrics=metrics, info=info)

    def step(self, state, action):
        info = state.info
        t = info["t"]
        action = jnp.clip(action, -1.0, 1.0)
        cmd = core.command_at(jnp, info["cmd_seq"], info["cmd_t"], t)
        action = core.park_brake(jnp, action, cmd)
        data = state.pipeline_state
        q_lock = jnp.where(info["fail_t"][:core.N_JOINTS] == t, data.qpos[self.qadr], info["q_lock"])
        mode = core.active_modes(jnp, info["mode"], info["fail_t"], t)
        model = self.mjx_model
        if self.randomize:
            model = model.replace(geom_friction=model.geom_friction.at[:, 0].set(info["friction"]),
                                  geom_solref=model.geom_solref.at[:, 0].set(info["softness"]))

        def substep(data, _):
            tau = core.actuator_torque(jnp, action, data.qpos[self.qadr], data.qvel[self.vadr], data.qvel[self.wadr],
                                       mode, info["strength"], q_lock)
            return mjx.step(model, data.replace(ctrl=tau)), tau

        data, taus = jax.lax.scan(substep, data, None, length=core.N_SUBSTEPS)
        qpos, qvel = data.qpos, data.qvel
        scan = core.scan_heights(jnp, self.terrain, qpos)
        ground = core.ground_level(jnp, scan)
        normal = core.ground_normal(jnp, self.terrain, qpos[0], qpos[1])
        gaps = core.foot_gaps(jnp, self.terrain, data.geom_xpos[self.contact], self.contact_r)
        contact = gaps < core.CONTACT_TOL
        air_time, landed, first = core.air_time_step(jnp, info["air_time"], info["last_contact"], contact)
        phase = core.gait_step(jnp, info["phase"], cmd)
        terms = core.reward_terms(jnp, qpos, qvel, self.qadr, self.vadr, self.wadr, cmd, action, info["last_action"],
                                  taus[-1], mode, ground, normal, contact, landed, first, phase, gaps)
        reward = sum(terms.values())
        t_next = t + 1
        vel_noise = 0.0
        if self.randomize:
            vel_noise = core.VEL_NOISE * jax.random.normal(jax.random.fold_in(info["rng"], t), (3,))
        actor_now = core.actor_obs(jnp, qpos, qvel, self.qadr, self.vadr, self.wadr, action,
                                   core.command_at(jnp, info["cmd_seq"], info["cmd_t"], t_next), phase, vel_noise)
        history = jnp.concatenate([state.obs["state"][core.ACTOR_OBS:core.HIST_SIZE], actor_now])
        obs = self._obs(history, actor_now, qpos, qvel, core.active_modes(jnp, info["mode"], info["fail_t"], t_next),
                        info["strength"], contact, air_time, scan, ground, normal)
        done = core.fallen(jnp, qpos, ground) | ~jnp.isfinite(obs["state"]).all() | ~jnp.isfinite(reward)
        yaw = core.heading(jnp, qpos[3:7])
        c, s = jnp.cos(yaw), jnp.sin(yaw)
        cmd_disp = info["cmd_disp"] + core.CTRL_DT * jnp.stack([c * cmd[0] - s * cmd[1], s * cmd[0] + c * cmd[1]])
        vel = core.body_velocity(jnp, qpos, qvel)
        wheel_working = (mode[core.N_JOINTS:] == core.HEALTHY) | (mode[core.N_JOINTS:] == core.WEAK)
        metrics = dict(state.metrics)
        metrics.update({f"r_{key}": jnp.nan_to_num(value) for key, value in terms.items()})
        metrics.update(
            vx=jnp.nan_to_num(vel[0]), vy=jnp.nan_to_num(vel[1]), wz=jnp.nan_to_num(vel[2]),
            lin_err=jnp.nan_to_num(jnp.linalg.norm(vel[:2] - cmd[:2])),
            yaw_err=jnp.nan_to_num(jnp.abs(vel[2] - cmd[2])),
            failed=jnp.sum(mode != core.HEALTHY).astype(jnp.float32),
            wheels_dead=jnp.all(~wheel_working).astype(jnp.float32),
            level_per_step=info["level"].astype(jnp.float32),
        )
        info = {**info, "t": t_next, "q_lock": q_lock, "last_action": action, "air_time": air_time,
                "last_contact": contact, "cmd_disp": cmd_disp, "phase": phase}
        return state.replace(pipeline_state=data, obs=obs, reward=jnp.nan_to_num(reward),
                             done=done.astype(jnp.float32), metrics=metrics, info=info)

    def _obs(self, history, actor_now, qpos, qvel, mode, strength, contact, air_time, scan=None, ground=None,
             normal=None):
        if scan is None:
            scan = core.scan_heights(jnp, self.terrain, qpos)
            ground = core.ground_level(jnp, scan)
            normal = core.ground_normal(jnp, self.terrain, qpos[0], qpos[1])
        scan_o = core.scan_obs(jnp, qpos, scan)
        return {"state": jnp.concatenate([history, scan_o]),
                "privileged_state": core.critic_obs(jnp, actor_now, scan_o, qpos, qvel, ground, normal, mode, strength,
                                                    contact, air_time)}

    def _sample_commands(self, rng, level):
        k_hold, k_cmd, k_stand = jax.random.split(rng, 3)
        holds = jax.random.randint(k_hold, (core.CMD_SEGMENTS,), *core.CMD_HOLD)
        cmd_t = jnp.concatenate([jnp.zeros(1, jnp.int32), jnp.cumsum(holds)[:-1].astype(jnp.int32)])
        high = jnp.asarray(core.CMD_HIGH, jnp.float32).at[0].add(-core.SPEED_DROP * level)
        cmd = core.CMD_LOW + jax.random.uniform(k_cmd, (core.CMD_SEGMENTS, 3)) * (high - core.CMD_LOW)
        stand = jax.random.uniform(k_stand, (core.CMD_SEGMENTS,)) < core.P_STAND
        return jnp.where(stand[:, None], 0.0, cmd), cmd_t

    def _sample_failures(self, rng):
        k_kind, k_legs, k_motor, k_a, k_b, k_w = jax.random.split(rng, 6)
        kind = jnp.minimum(jnp.searchsorted(jnp.asarray(core.FAIL_CUM), jax.random.uniform(k_kind), side="right"),
                           len(core.FAIL_P) - 1)
        legs = jax.random.permutation(k_legs, core.N_LEGS)
        motor = jax.random.randint(k_motor, (), 0, core.N_ACT)
        leg_a = (kind == core.ONE_LEG) | (kind == core.TWO_LEGS) | (kind == core.WHEELS_AND_LEG)
        in_a = jnp.where(kind == core.ONE_MOTOR, jnp.arange(core.N_ACT) == motor, leg_a & (core.ACT_LEG == legs[0]))
        in_b = (kind == core.TWO_LEGS) & (core.ACT_LEG == legs[1])
        in_w = (kind >= core.ALL_WHEELS) & core.IS_WHEEL

        def when(key):
            k_now, k_time = jax.random.split(key)
            return jnp.where(jax.random.uniform(k_now) < 0.5, 0,
                             jax.random.randint(k_time, (), core.FAIL_EARLIEST, core.FAIL_LATEST))

        def group(key):
            k_mode, k_strength, k_when = jax.random.split(key, 3)
            v = jax.random.uniform(k_mode)
            mode = jnp.where(v < core.P_LIMP, core.LIMP, jnp.where(v < core.P_LIMP + core.P_LOCKED, core.LOCKED, core.WEAK))
            strength = jnp.where(mode == core.WEAK, jax.random.uniform(k_strength, minval=core.WEAK_RANGE[0],
                                                                      maxval=core.WEAK_RANGE[1]), 1.0)
            return mode, strength, when(k_when)

        (mode_a, strength_a, when_a), (mode_b, strength_b, when_b) = group(k_a), group(k_b)
        k_wmode, k_wwhen = jax.random.split(k_w)
        mode_w = jnp.where(jax.random.uniform(k_wmode) < core.P_WHEELS_LIMP, core.LIMP, core.LOCKED)
        when_w = when(k_wwhen)
        mode = jnp.where(in_w, mode_w, jnp.where(in_a, mode_a, jnp.where(in_b, mode_b, core.HEALTHY))).astype(jnp.int32)
        strength = jnp.where(in_w, 1.0, jnp.where(in_a, strength_a, jnp.where(in_b, strength_b, 1.0)))
        fail_t = jnp.where(in_w, when_w, jnp.where(in_a, when_a, jnp.where(in_b, when_b, core.NEVER))).astype(jnp.int32)
        return mode, strength, fail_t


class FullResetWrapper(Wrapper):
    def step(self, state, action):
        if "steps" in state.info:
            state.info["steps"] = jnp.where(state.done, jnp.zeros_like(state.info["steps"]), state.info["steps"])
        state = self.env.step(state.replace(done=jnp.zeros_like(state.done)), action)
        base = self.env.unwrapped
        info = dict(state.info)
        keys = jax.vmap(lambda key: jax.random.split(key, 3))(info["rng"])
        level = info["level"]
        if base.curriculum:
            done = state.done > 0
            fell = done & (info.get("truncation", jnp.zeros_like(state.done)) == 0)
            moved = jnp.linalg.norm(state.pipeline_state.qpos[:, :2] - info["spawn_xy"], axis=-1)
            asked = jnp.linalg.norm(info["cmd_disp"], axis=-1)
            up = done & ~fell & (moved > core.PROMOTE_DIST)
            down = done & ~up & (fell | (moved < core.DEMOTE_RATIO * asked))
            new = level + up.astype(jnp.int32) - down.astype(jnp.int32)
            anywhere = jax.vmap(lambda key: jax.random.randint(key, (), 0, core.LEVELS))(keys[:, 2])
            level = jnp.where(new >= core.LEVELS, anywhere, jnp.maximum(new, 0))
            histogram = jnp.mean(jax.nn.one_hot(level, core.LEVELS), axis=0)
            report = jax.random.uniform(jax.random.fold_in(keys[0, 2], 7)) < 1.0 / 64
            jax.lax.cond(report, lambda h: jax.debug.callback(_record_levels, h), lambda h: None, histogram)
        fresh = jax.vmap(base.reset_at)(keys[:, 1], level, info["column"])

        def pick(new, old):
            done = state.done.reshape(state.done.shape + (1,) * (old.ndim - state.done.ndim))
            return jnp.where(done > 0, new, old)

        for key in INFO_KEYS:
            info[key] = pick(fresh.info[key], info[key])
        info["rng"] = pick(fresh.info["rng"], keys[:, 0])
        return state.replace(
            pipeline_state=jax.tree_util.tree_map(pick, fresh.pipeline_state, state.pipeline_state),
            obs=jax.tree_util.tree_map(pick, fresh.obs, state.obs),
            info=info,
        )


def wrap_for_training(env, episode_length=core.EPISODE_STEPS, action_repeat=1, randomization_fn=None, **_):
    return FullResetWrapper(VmapWrapper(EpisodeWrapper(env, episode_length, action_repeat)))

In [ ]:
%%writefile hexapod/policy.py
"""Runs the exported policy (policy.npz) with NumPy only."""
import numpy as np


class Policy:
    def __init__(self, path="policy.npz"):
        f = np.load(path)
        self.mean, self.std = f["obs_mean"], f["obs_std"]
        self.act_size = int(f["act_size"])
        self.layers = [(f[f"w{i}"], f[f"b{i}"]) for i in range(int(f["n_layers"]))]

    def act(self, obs):
        x = (np.asarray(obs, np.float32) - self.mean) / self.std
        with np.errstate(over="ignore"):
            for i, (w, b) in enumerate(self.layers):
                x = x @ w + b
                if i < len(self.layers) - 1:
                    x = x / (1.0 + np.exp(-x))
        return np.tanh(x[: self.act_size])

In [ ]:
%%writefile train.py
"""Trains the policy with Brax PPO on MJX.

    python train.py --steps 300_000_000 --video
"""
import argparse
import functools
import inspect
import sys
import time
from pathlib import Path

import jax
import numpy as np
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo

from hexapod import core, mjx_env
from hexapod.env import HexapodEnv
from hexapod.mjx_env import HexapodMJX, wrap_for_training
from hexapod.policy import Policy

HIDDEN = (512, 256, 128)
PPO_CONFIG = dict(
    num_envs=4096, num_eval_envs=128, batch_size=256, num_minibatches=32, unroll_length=20,
    num_updates_per_batch=4, learning_rate=3e-4, entropy_cost=1e-2, discounting=0.97,
    gae_lambda=0.95, clipping_epsilon=0.2, max_grad_norm=1.0, normalize_observations=True,
    reward_scaling=1.0, episode_length=core.EPISODE_STEPS,
)
SMOKE_CONFIG = dict(
    num_timesteps=4_000, num_evals=2, num_envs=8, num_eval_envs=8, batch_size=8,
    num_minibatches=4, unroll_length=10, num_updates_per_batch=1, episode_length=60,
)
FLAT = ((-3.0, 0.0), np.pi)
SCENARIOS = (
    ("flat, forward 0.6", (0.6, 0.0, 0.0), None, FLAT),
    ("flat, forward 0.3 (middle legs in rhythm)", (0.3, 0.0, 0.0), None, FLAT),
    ("flat, forward 1.2 (wheels + steps)", (1.2, 0.0, 0.0), None, FLAT),
    ("flat, turn in place 1.0 rad/s", (0.0, 0.0, 1.0), None, FLAT),
    ("flat, forward 0.3 + turn 1.0", (0.3, 0.0, 1.0), None, FLAT),
    ("flat, sideways 0.15", (0.0, 0.15, 0.0), None, FLAT),
    ("flat, wheels locked, 0.4", (0.4, 0.0, 0.0), ("wheels", core.LOCKED), FLAT),
    ("flat, wheels limp, 0.4", (0.4, 0.0, 0.0), ("wheels", core.LIMP), FLAT),
    ("flat, LF leg limp, 0.6", (0.6, 0.0, 0.0), ("leg", 0, core.LIMP), FLAT),
    ("ramp 6 from the edge (climb), 0.5", (0.5, 0.0, 0.0), None, ("ramp", 6, "edge")),
    ("rubble 5, 0.5", (0.5, 0.0, 0.0), None, ("rubble", 5)),
    ("blocks 5, 0.5", (0.5, 0.0, 0.0), None, ("blocks", 5)),
    ("stairs 5 from the edge (climb), 0.4", (0.4, 0.0, 0.0), None, ("stairs", 5, "edge")),
    ("stairs 5 from the top (descent), 0.4", (0.4, 0.0, 0.0), None, ("stairs", 5)),
    ("trenches 5, 0.4", (0.4, 0.0, 0.0), None, ("trenches", 5)),
    ("stairs 3 from the edge, wheels locked, 0.3", (0.3, 0.0, 0.0), ("wheels", core.LOCKED), ("stairs", 3, "edge")),
)


def export_policy(params, path, quiet=False):
    normalizer, policy_params = params[0], params[1]
    mean, std = normalizer.mean, normalizer.std
    if isinstance(mean, dict):
        mean, std = mean["state"], std["state"]
    layers = policy_params["params"] if "params" in policy_params else policy_params
    names = sorted(layers, key=lambda name: int(name.rsplit("_", 1)[-1]))
    arrays = {"obs_mean": mean, "obs_std": std}
    for i, name in enumerate(names):
        arrays[f"w{i}"], arrays[f"b{i}"] = layers[name]["kernel"], layers[name]["bias"]
    arrays = {key: np.asarray(value, np.float32) for key, value in arrays.items()}
    np.savez(path, act_size=core.N_ACT, n_layers=len(names), **arrays)
    if not quiet:
        print(f"wrote {path}; layers {[arrays[f'w{i}'].shape for i in range(len(names))]}")


CHECKPOINT_CONFIG = "ppo_network_config.json"


def latest_checkpoint(root):
    root = Path(root)
    found = [p.parent for p in root.rglob(CHECKPOINT_CONFIG) if p.parent.name.isdigit() and len(p.parent.name) == 12]
    if (root / CHECKPOINT_CONFIG).exists():
        found.append(root)
    if not found:
        sys.exit(f"no checkpoint under {root} (looking for checkpoints/<12-digit step>/{CHECKPOINT_CONFIG})")
    return max(found, key=lambda p: int(p.name))


def spawn(env, place, seed):
    if place == FLAT or place is None:
        (x, y), yaw = FLAT
        return env.reset(seed=seed, pos=(x, y), yaw=yaw)
    kind, level, *where = place
    if where:
        cx, cy = core.tile_center(level, core.TERRAIN_TYPES.index(kind))
        return env.reset(seed=seed, pos=(cx - core.EDGE_OFFSET, cy), yaw=0.0)
    return env.reset(seed=seed, level=level, column=kind, yaw=0.0)


def apply_failure(env, failure):
    kind, *rest = failure
    if kind == "wheels":
        env.fail_wheels(rest[0])
    elif kind == "leg":
        env.fail_leg(rest[0], rest[1])
    else:
        env.fail_actuators([rest[0]], rest[1])


def evaluate(path, episodes=3, seconds=10.0, fail_at=3.0):
    policy = Policy(path)
    steps, t_fail, window = int(seconds / core.CTRL_DT), int(fail_at / core.CTRL_DT), int(3.0 / core.CTRL_DT)
    print(f"Sim2sim (CPU MuJoCo), {seconds:.0f} s, failures at {fail_at:.0f} s; velocity: mean of the last 3 s, "
          f"progress: distance along the initial heading:")
    print(f"  {'scenario':44s}   vx     vy     ωz   (command)             progress  falls")
    for label, cmd, failure, place in SCENARIOS:
        env = HexapodEnv(failures=False)
        env.set_command(cmd)
        means, progress, falls = [], [], 0
        for episode in range(episodes):
            obs, _ = spawn(env, place, episode)
            start, yaw = env.data.qpos[:2].copy(), core.heading(np, env.data.qpos[3:7])
            trace = []
            for t in range(steps):
                if failure and t == t_fail:
                    apply_failure(env, failure)
                obs, _, terminated, _, info = env.step(policy.act(obs))
                trace.append(info["velocity"])
                if terminated:
                    falls += 1
                    break
            means.append(np.mean(trace[-window:], axis=0))
            progress.append(np.dot(env.data.qpos[:2] - start, (np.cos(yaw), np.sin(yaw))))
        v = np.mean(means, axis=0)
        print(f"  {label:44s} {v[0]:+.2f}  {v[1]:+.2f}  {v[2]:+.2f}   ({cmd[0]:+.2f} {cmd[1]:+.2f} {cmd[2]:+.2f})"
              f"   {np.mean(progress):5.1f} m    {falls}/{episodes}")


def record_video(path, out="hexapod.mp4"):
    import mediapy

    policy = Policy(path)
    segments = (
        ("flat ground", FLAT, ((0.0, (0.3, 0.0, 0.0)), (3.0, (1.2, 0.0, 0.0)), (6.0, (0.0, 0.0, 1.0)),
                             (8.5, (0.0, 0.0, 0.0))), None, 10.0),
        ("stairs 4, from the edge", ("stairs", 4, "edge"), ((0.0, (0.4, 0.0, 0.0)),), None, 8.0),
        ("trenches 4, wheels locked", ("trenches", 4), ((0.0, (0.4, 0.0, 0.0)),), ("wheels", core.LOCKED), 8.0),
    )
    frames = []
    for label, place, program, failure, seconds in segments:
        env = HexapodEnv(failures=False)
        env.set_command(program[0][1])
        obs, _ = spawn(env, place, 0)
        for t in range(int(seconds / core.CTRL_DT)):
            for at, cmd in program:
                if t == int(at / core.CTRL_DT):
                    env.set_command(cmd)
            if failure and t == int(2.0 / core.CTRL_DT):
                apply_failure(env, failure)
            obs, _, terminated, _, _ = env.step(policy.act(obs))
            if t % 2 == 0:
                frames.append(env.render(distance=1.0, elevation=-20.0))
            if terminated:
                print(f"  video: fell during '{label}' ({t * core.CTRL_DT:.1f} s)")
                break
        env.close()
    mediapy.write_video(out, frames, fps=round(0.5 / core.CTRL_DT))
    print(f"wrote {out} ({len(frames)} frames): " + " | ".join(s[0] for s in segments))


def main():
    sys.stdout.reconfigure(encoding="utf-8")
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--steps", type=int, default=300_000_000)
    parser.add_argument("--evals", type=int, default=10)
    parser.add_argument("--envs", type=int, default=PPO_CONFIG["num_envs"])
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--out", default="policy.npz")
    parser.add_argument("--checkpoints", default="checkpoints",
                        help="folder for full training checkpoints (empty: do not save)")
    parser.add_argument("--resume", default=None,
                        help="checkpoints folder of a previous run (or a folder containing it); resumes from the latest step")
    parser.add_argument("--video", action="store_true", help="record hexapod.mp4 after training")
    parser.add_argument("--smoke", action="store_true", help="few-minute end-to-end test on CPU")
    args = parser.parse_args()

    print("JAX devices:", jax.devices(), flush=True)
    cfg = {**PPO_CONFIG, "num_timesteps": args.steps, "num_evals": args.evals, "num_envs": args.envs, "seed": args.seed}
    if args.smoke:
        cfg.update(SMOKE_CONFIG)
    cfg["policy_params_fn"] = lambda step, make_policy, params: export_policy(params, args.out, quiet=True)
    if args.checkpoints:
        cfg["save_checkpoint_path"] = str(Path(args.checkpoints).resolve())
    if args.resume:
        cfg["restore_checkpoint_path"] = str(latest_checkpoint(args.resume))
        print(f"Resuming from {cfg['restore_checkpoint_path']} (policy, critic and normalizer)", flush=True)

    accepted = inspect.signature(ppo.train).parameters
    env = HexapodMJX(curriculum=True, init_level_max=core.LEVELS - 1 if args.resume else core.INIT_LEVEL_MAX)
    eval_env = HexapodMJX(curriculum=False)
    if "wrap_env_fn" in accepted:
        cfg["wrap_env_fn"] = wrap_for_training
    else:
        env = wrap_for_training(env, cfg["episode_length"])
        eval_env = wrap_for_training(eval_env, cfg["episode_length"])
        cfg["wrap_env"] = False
    skipped = sorted(set(cfg) - set(accepted))
    if skipped:
        print("Settings not supported by this Brax version, skipped:", skipped)

    network = functools.partial(ppo_networks.make_ppo_networks, policy_hidden_layer_sizes=HIDDEN,
                                value_hidden_layer_sizes=HIDDEN, policy_obs_key="state",
                                value_obs_key="privileged_state")
    start = time.time()

    def progress(step, metrics):
        length = max(float(metrics.get("eval/avg_episode_length", 1.0)), 1.0)
        mean = lambda key: float(metrics.get(f"eval/episode_{key}", float("nan"))) / length
        levels = mjx_env.CURRICULUM["levels"]
        print(f"{step:>13,} steps | reward {float(metrics.get('eval/episode_reward', float('nan'))):7.1f} "
              f"| velocity error {mean('lin_err'):.3f} m/s | yaw error {mean('yaw_err'):.2f} rad/s "
              f"| episode {length:4.0f} | curriculum level mean {np.dot(levels, np.arange(core.LEVELS)):.2f}, "
              f"top {100 * levels[-1]:.0f}% | {time.time() - start:5.0f} s", flush=True)

    make_inference_fn, params, _ = ppo.train(
        environment=env, eval_env=eval_env, network_factory=network, progress_fn=progress,
        **{key: value for key, value in cfg.items() if key in accepted})
    print(f"Training finished: {time.time() - start:.0f} s", flush=True)

    export_policy(params, args.out)
    inference = jax.jit(make_inference_fn(params, deterministic=True))
    k1, k2 = jax.random.split(jax.random.PRNGKey(1))
    obs = {"state": jax.random.normal(k1, (core.OBS_SIZE,)), "privileged_state": jax.random.normal(k2, (core.CRITIC_OBS,))}
    reference = np.asarray(inference(obs, jax.random.PRNGKey(0))[0])
    gap = float(np.abs(reference - Policy(args.out).act(np.asarray(obs["state"]))).max())
    print(f"Brax vs numpy policy difference: {gap:.1e}" + ("" if gap < 1e-3 else "   <-- WARNING: export is wrong"))

    evaluate(args.out, episodes=1 if args.smoke else 3, seconds=2.0 if args.smoke else 10.0,
             fail_at=1.0 if args.smoke else 3.0)
    if args.video:
        try:
            record_video(args.out)
        except Exception as error:
            print(f"Could not record the video ({type(error).__name__}: {error})")


if __name__ == "__main__":
    main()

In [ ]:
# Robot model and terrain; the printed digest should be eca000310362.
!python -m hexapod.model

In [ ]:
%env MUJOCO_GL=egl
!python -c "import jax, jax.numpy as jnp; print(jax.__version__, jax.devices()); print('GPU test:', float(jnp.linalg.cholesky(2 * jnp.eye(3)).sum()))"

In [ ]:
!python train.py --steps 300_000_000 --video

In [ ]:
from pathlib import Path
from IPython.display import Video
Video("hexapod.mp4", embed=True, width=640) if Path("hexapod.mp4").exists() else print("no video")